# Developer Salary Prediction — Final Model

**Goal.** Predict annual developer compensation (`annual.pay.usd`) on the held-out test set. Metric: **RMSE**.

**Best model.** SVR (RBF kernel, C=1, ε=0.03, γ=auto) trained on salary range [1 k, 300 k]  
with smoothed K-fold target encoding for 8 high-cardinality columns  
and multi-hot encoding of multi-select survey fields (dev.tools excluded).

Parameters were selected via the exploration notebook (`salary_exploration.ipynb`).
Expected CV RMSE ≈ 29 600 (on [1 k, 200 k] eval); leaderboard ≈ 33–34 k.


## 1. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.svm import SVR

RNG = 42


## 2. Data Loading

In [ ]:
TRAIN_PATH = 'train.csv'
TEST_PATH  = 'test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

target = 'annual.pay.usd'
id_col = 'id'
role_col = 'dev.role'

# All multi-select survey columns (dev.tools excluded — adds noise, hurts CV)
multi_cols = [
    'dev.role',
    'side.coding',
    'how.learned.coding',
    'prog.languages',
    'databases',
    'cloud.platforms',
    'web.frameworks',
    'other.tech',
    'dev.environments',
    'personal.os',
    'work.os',
    'project.mgmt.tools',
    'comm.tools',
    'ai.search.tools',
    'ai.tools.used',
]

print('train:', train.shape, '| test:', test.shape)


## 3. Feature Engineering

In [ ]:
def fit_multiselect(df, cols, sep=';', min_freq=5, top_k=None):
    """Learn which category values to keep for each multi-select column."""
    categories = {}
    for col in cols:
        values = []
        for v in df[col].fillna(''):
            if col == role_col:
                items = re.split(r',(?![^()]*\))', str(v))
            else:
                items = str(v).split(sep)
            values.extend([i.strip() for i in items if i.strip() != ''])
        counts = pd.Series(values).value_counts()
        counts = counts[counts >= min_freq]
        if top_k is not None:
            counts = counts.head(top_k)
        categories[col] = list(counts.index)
    return categories


def transform_multiselect(df, categories, cols, sep=';',
                           add_count=True, drop_original=True):
    """Apply multi-hot encoding using pre-fitted categories."""
    df = df.copy()
    for col in cols:
        if col == role_col:
            split_values = df[col].fillna('').apply(
                lambda v: [i.strip() for i in re.split(r',(?![^()]*\))', str(v))
                           if i.strip() != '']
            )
        else:
            split_values = df[col].fillna('').apply(
                lambda v: [i.strip() for i in str(v).split(sep) if i.strip() != '']
            )
        for cat in categories[col]:
            df[f'{col}_{cat}'] = split_values.apply(lambda vals: int(cat in vals))
        if add_count:
            df[f'{col}_count'] = split_values.apply(len)
    if drop_original:
        df = df.drop(columns=cols)
    return df


def engineer(df, multi_categories, multi_cols):
    """Apply multi-hot encoding and add derived numeric features."""
    df = df.copy()
    df = transform_multiselect(
        df=df, categories=multi_categories, cols=multi_cols,
        sep=';', add_count=True, drop_original=True,
    )
    if 'coding.years.total' in df.columns and 'coding.years.professional' in df.columns:
        df['coding.years.non_professional'] = (
            df['coding.years.total'] - df['coding.years.professional']
        )
    if 'coding.years.professional' in df.columns and 'coding.years.total' in df.columns:
        df['professional_experience_share'] = (
            df['coding.years.professional']
            / df['coding.years.total'].replace(0, np.nan)
        )
    count_cols = [c for c in df.columns if c.endswith('_count')]
    if count_cols:
        df['total_multiselect_count'] = df[count_cols].sum(axis=1)
    return df


## 4. Preprocessing

In [ ]:
def make_preprocessor(X, id_col, num_imputer='mean',
                       cat_imputer='missing', scaler='standard'):
    """Build a ColumnTransformer for numeric + categorical features."""
    CAT_COLS = X.select_dtypes(include='object').columns.tolist()
    NUM_COLS = [c for c in X.columns if c not in CAT_COLS + [id_col]]

    if num_imputer in ['mean', 'median', 'most_frequent']:
        num_imp = SimpleImputer(strategy=num_imputer)
    elif num_imputer == 'constant_0':
        num_imp = SimpleImputer(strategy='constant', fill_value=0)
    else:
        raise ValueError(f'Unknown num_imputer: {num_imputer}')

    sc = StandardScaler() if scaler == 'standard' else None
    num_steps = [('imp', num_imp)]
    if sc is not None:
        num_steps.append(('sc', sc))
    num_pipe = Pipeline(num_steps)

    cat_imp = (
        SimpleImputer(strategy='constant', fill_value='missing')
        if cat_imputer == 'missing'
        else SimpleImputer(strategy='most_frequent')
    )
    cat_pipe = Pipeline([
        ('imp', cat_imp),
        ('oh', OneHotEncoder(handle_unknown='ignore')),
    ])

    preprocessor = ColumnTransformer([
        ('num', num_pipe, NUM_COLS),
        ('cat', cat_pipe, CAT_COLS),
    ])
    return preprocessor, NUM_COLS, CAT_COLS


def kfold_target_encode(X_tr, y_tr_log, X_val, cols,
                         n_splits=5, smoothing=20, random_state=42):
    """Out-of-fold smoothed target encoding (leakage-safe)."""
    X_tr  = X_tr.copy().reset_index(drop=True)
    X_val = X_val.copy().reset_index(drop=True)
    y_arr = pd.Series(np.asarray(y_tr_log), index=X_tr.index)
    global_mean = float(y_arr.mean())

    for col in cols:
        if col not in X_tr.columns:
            continue
        s = X_tr[col].fillna('__missing__').astype(str)

        oof = pd.Series(np.nan, index=X_tr.index, dtype=float)
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        for tr_i, va_i in kf.split(X_tr):
            agg = y_arr.iloc[tr_i].groupby(s.iloc[tr_i]).agg(['mean', 'count'])
            sm  = (agg['mean'] * agg['count'] + global_mean * smoothing) \
                  / (agg['count'] + smoothing)
            oof.iloc[va_i] = s.iloc[va_i].map(sm).fillna(global_mean).values
        X_tr[f'{col}_te'] = oof.fillna(global_mean).values

        agg_full = y_arr.groupby(s).agg(['mean', 'count'])
        sm_full  = (agg_full['mean'] * agg_full['count'] + global_mean * smoothing) \
                   / (agg_full['count'] + smoothing)
        s_val = X_val[col].fillna('__missing__').astype(str)
        X_val[f'{col}_te'] = s_val.map(sm_full).fillna(global_mean).values

    return X_tr, X_val


## 5. Final Model Training & Prediction

### Hyperparameters

| Parameter | Value | Chosen by |
| --- | --- | --- |
| Kernel | RBF | model comparison (Sec 5) |
| C | 1 | SVR grid search (Sec 9.2) |
| epsilon | 0.03 | SVR grid search (Sec 9.2) |
| gamma | auto | SVR grid search (Sec 9.2) |
| Salary filter | [1 000, 300 000] | range sweep (Sec 9.4) |
| min_freq / top_k | 100 / 10 | multi-select tuning (Sec 6) |
| Target encoding | 8 columns | ablation (Sec 9.1) |
| Imputer / scaler | mean / standard | preprocessing tuning (Sec 6) |


In [ ]:
# ── Tuned hyperparameters (do not change without re-running exploration) ─────
SALARY_MIN  = 1_000
SALARY_MAX  = 300_000
MIN_FREQ    = 100
TOP_K       = 10
NUM_IMPUTER = 'mean'
SCALER      = 'standard'

# Target-encoding columns (extended set from Section 9 ablation)
TE_COLS = (
    'region', 'industry', 'company.size', 'age.group',
    'dev.role', 'education', 'employment.type', 'work.location',
)

model = SVR(kernel='rbf', C=1.0, epsilon=0.03, gamma='auto')
# ─────────────────────────────────────────────────────────────────────────────


def fit_predict_full(model, train_df, test_df):
    """Filter train, fit the pipeline, return test predictions in USD."""
    df = train_df[
        (train_df[target] >= SALARY_MIN) & (train_df[target] <= SALARY_MAX)
    ].copy()
    X_tr_raw = df.drop(columns=[target]).reset_index(drop=True)
    y_log    = np.log(df[target]).reset_index(drop=True)
    X_te_raw = test_df.drop(columns=[id_col]).reset_index(drop=True)

    cats = fit_multiselect(X_tr_raw, multi_cols, min_freq=MIN_FREQ, top_k=TOP_K)
    X_tr = engineer(X_tr_raw, cats, multi_cols)
    X_te = engineer(X_te_raw, cats, multi_cols)

    valid_te = [c for c in TE_COLS if c in X_tr.columns]
    X_tr, X_te = kfold_target_encode(
        X_tr, y_log, X_te, valid_te,
        n_splits=5, smoothing=20, random_state=RNG,
    )

    pre, _, _ = make_preprocessor(
        X_tr, id_col=id_col,
        num_imputer=NUM_IMPUTER, scaler=SCALER,
    )
    pipe = Pipeline([('pre', pre), ('model', clone(model))])
    pipe.fit(X_tr, y_log)
    return np.exp(pipe.predict(X_te))


print('Fitting model on filtered training data...')
test_pred = fit_predict_full(model, train, test)

submission = pd.DataFrame({id_col: test[id_col].values, target: test_pred})
submission.to_csv('submission.csv', index=False)
print(f'Wrote submission.csv {submission.shape}')
print(f'Prediction stats — mean: {test_pred.mean():,.0f}  '
      f'median: {np.median(test_pred):,.0f}  '
      f'min: {test_pred.min():,.0f}  max: {test_pred.max():,.0f}')
submission.head()
